<a href="https://colab.research.google.com/github/khu3086/surprise-architecture/blob/main/lewm_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 SURPRISE — LeWorldModel Training Starter

> Companion notebook for the [SURPRISE deepfake detection project](https://github.com/khu3086/surprise-deepfake-detector).

This notebook gets you from **zero** to **first training step** on free Colab GPU. Run cells top to bottom.

## What this notebook does

1. Verifies you have a GPU
2. Installs PyTorch + dependencies
3. Downloads a tiny video dataset to test with
4. Builds a *minimal* LeWorldModel-style architecture (encoder + predictor)
5. Runs one training step end-to-end so you know the pipeline works
6. Tells you what to do next

## What this notebook does NOT do

- Train a production model (that needs FaceForensics++ and many hours of GPU)
- Use the official LeWM codebase (we build a minimal version for clarity)
- Implement SIGReg properly (we use a simpler placeholder regularizer)

Think of this as **scaffolding** — once it runs end-to-end, you swap in the real components piece by piece.

## Before running

**Enable GPU:** Runtime → Change runtime type → Hardware accelerator: **T4 GPU** → Save.

If you skip this step, training will run on CPU and take ~50× longer.

## 1. GPU sanity check

First thing: confirm you actually have a GPU. If this prints `cuda` you're good. If it prints `cpu`, go to Runtime → Change runtime type and pick T4.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type and pick T4 GPU.')
    print('   Continuing on CPU — training will be slow but functional.')

## 2. Install dependencies

Colab comes with PyTorch preinstalled. We just need a few extras for video handling and the training utilities. The `-q` flag keeps the output quiet.

In [ ]:
!pip install -q einops opencv-python-headless tqdm matplotlib

In [ ]:
# Import everything we'll use throughout the notebook
import os
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from einops import rearrange
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('✓ Imports OK')

## 3. Generate a tiny synthetic dataset

**Why synthetic?** FaceForensics++ requires an academic email and a download form, and we want this notebook to *just work* without external dependencies. We'll generate videos of bouncing colored squares — simple enough to train on in seconds, but with real motion dynamics.

Once your training loop works on synthetic data, swapping in real video data is a one-class change.

In [ ]:
def make_synthetic_clip(n_frames=16, size=64, n_objects=2, perturbed=False):
    """
    Generate a video of bouncing squares.

    Args:
        n_frames: how many frames
        size: spatial resolution (HxW)
        n_objects: how many bouncing squares
        perturbed: if True, teleport an object mid-clip (mimics deepfake artifacts)

    Returns:
        np.ndarray of shape (n_frames, size, size, 3) uint8 RGB
    """
    frames = np.zeros((n_frames, size, size, 3), dtype=np.uint8)

    # Initialize random object positions, velocities, colors
    positions = np.random.uniform(8, size - 8, size=(n_objects, 2))
    velocities = np.random.uniform(-3, 3, size=(n_objects, 2))
    colors = np.random.randint(80, 255, size=(n_objects, 3))

    for t in range(n_frames):
        frame = np.zeros((size, size, 3), dtype=np.uint8)

        for i in range(n_objects):
            x, y = positions[i]
            # Inject a teleportation halfway through (the "fake" signal)
            if perturbed and t == n_frames // 2 and i == 0:
                positions[i] = np.random.uniform(8, size - 8, size=2)
                x, y = positions[i]

            # Draw the square
            x0, y0 = int(max(0, x - 4)), int(max(0, y - 4))
            x1, y1 = int(min(size, x + 4)), int(min(size, y + 4))
            frame[y0:y1, x0:x1] = colors[i]

            # Update position with bounce
            positions[i] += velocities[i]
            for d in range(2):
                if positions[i][d] < 4 or positions[i][d] > size - 4:
                    velocities[i][d] *= -1
                    positions[i][d] = np.clip(positions[i][d], 4, size - 4)

        frames[t] = frame

    return frames


# Quick visual test — show a few frames from one clip
test_clip = make_synthetic_clip(n_frames=8, size=64)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(test_clip[i])
    ax.set_title(f't={i}', fontsize=10)
    ax.axis('off')
plt.suptitle('Synthetic training clip — bouncing squares', y=1.05)
plt.tight_layout()
plt.show()

## 4. Build a Dataset class

PyTorch wants data in a `Dataset`. Ours generates clips on the fly — no disk needed, no download needed.

In [ ]:
class SyntheticVideoDataset(Dataset):
    """Generates random bouncing-square videos on the fly."""

    def __init__(self, n_clips=200, n_frames=16, size=64):
        self.n_clips = n_clips
        self.n_frames = n_frames
        self.size = size

    def __len__(self):
        return self.n_clips

    def __getitem__(self, idx):
        clip = make_synthetic_clip(self.n_frames, self.size)
        # Convert to (T, C, H, W) float tensor in [0, 1]
        clip = torch.from_numpy(clip).float() / 255.0
        clip = rearrange(clip, 't h w c -> t c h w')
        return clip


dataset = SyntheticVideoDataset(n_clips=200)
loader = DataLoader(dataset, batch_size=16, num_workers=2, shuffle=True)

# Sanity check: grab one batch and check the shape
batch = next(iter(loader))
print(f'Batch shape: {batch.shape}')
print(f'  → (batch_size={batch.shape[0]}, n_frames={batch.shape[1]}, channels={batch.shape[2]}, h={batch.shape[3]}, w={batch.shape[4]})')
print(f'Value range: [{batch.min():.3f}, {batch.max():.3f}]')

## 5. Build the LeWM-style architecture

The full LeWorldModel uses a Vision Transformer + Transformer predictor. For this starter, we'll build a simpler version with the same conceptual shape:

- **Encoder:** small CNN that maps each frame → 64-dim embedding
- **Predictor:** small MLP/transformer that predicts the next embedding from the current one
- **No actions** (passive video, like deepfake detection)

This is intentionally tiny so it trains in seconds. Once the training loop works, you swap in larger / paper-faithful versions.

In [ ]:
class TinyEncoder(nn.Module):
    """
    A small CNN encoder. Maps (B, 3, 64, 64) → (B, embed_dim).

    In the full LeWM, this would be a Vision Transformer Tiny operating on
    patches of a 224×224 frame. The structural role is identical: image → vector.
    """

    def __init__(self, embed_dim=64):
        super().__init__()
        self.embed_dim = embed_dim
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # 64 → 32
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 32 → 16
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 16 → 8
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),  # 8 → 1
            nn.Flatten(),
            nn.Linear(128, embed_dim),
        )
        # BatchNorm on the embeddings — important for SIGReg in the real LeWM
        self.norm = nn.BatchNorm1d(embed_dim, affine=False)

    def forward(self, x):
        z = self.net(x)
        z = self.norm(z)
        return z


class TinyPredictor(nn.Module):
    """
    Predicts the next embedding from the current one. No actions.

    The full LeWM uses a multi-layer Transformer with causal masking and
    AdaLN action conditioning. We use a simple MLP for clarity.
    """

    def __init__(self, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, z_t):
        return self.net(z_t)


encoder = TinyEncoder(embed_dim=64).to(device)
predictor = TinyPredictor(embed_dim=64).to(device)

n_params = sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in predictor.parameters())
print(f'Encoder: {sum(p.numel() for p in encoder.parameters()):,} params')
print(f'Predictor: {sum(p.numel() for p in predictor.parameters()):,} params')
print(f'Total: {n_params:,} params')
print('(Real LeWM is ~15M; this tiny version is much smaller for fast iteration)')

## 6. The training step

This is the heart of LeWM. For each clip:

1. Encode all frames into latent embeddings: `z_t = encoder(o_t)`
2. Predict next-frame embeddings: `ẑ_{t+1} = predictor(z_t)`
3. Compute prediction loss: `MSE(ẑ_{t+1}, z_{t+1})`
4. Add an anti-collapse regularizer (placeholder for SIGReg)
5. Backprop

Without step 4, the encoder collapses (every frame → same vector, prediction is trivially perfect, embeddings become useless). The full LeWM uses SIGReg; we use a simpler covariance regularizer that has the same effect.

In [ ]:
def covariance_regularizer(z, eps=1e-6):
    """
    Simple anti-collapse: encourage embedding dimensions to be uncorrelated
    and unit-variance. A poor-man's SIGReg — keeps the embedding space spread out.

    Real LeWM uses SIGReg (random projections + Epps-Pulley test). For a starter,
    this is enough to prevent total collapse and lets us focus on the training loop.
    """
    z = z - z.mean(dim=0, keepdim=True)
    cov = (z.T @ z) / (z.shape[0] - 1)
    diag = torch.diagonal(cov)
    off_diag = cov - torch.diag(diag)
    # Variance should be ≈ 1, off-diagonals should be ≈ 0
    var_loss = ((diag - 1) ** 2).mean()
    cov_loss = (off_diag ** 2).sum() / z.shape[1]
    return var_loss + cov_loss


def lewm_loss(clips, encoder, predictor, lam=1.0):
    """
    Compute the LeWM-style training loss for a batch of video clips.

    Args:
        clips: (B, T, C, H, W) batch of clips
        encoder: image → embedding
        predictor: embedding_t → embedding_{t+1}
        lam: weight for anti-collapse regularizer

    Returns:
        total_loss, dict of components
    """
    B, T, C, H, W = clips.shape

    # 1. Encode every frame
    flat = rearrange(clips, 'b t c h w -> (b t) c h w')
    z = encoder(flat)
    z = rearrange(z, '(b t) d -> b t d', b=B)  # (B, T, D)

    # 2. Predict next embeddings: input z_{0..T-2}, target z_{1..T-1}
    z_in = z[:, :-1, :]
    z_target = z[:, 1:, :]
    z_pred = predictor(rearrange(z_in, 'b t d -> (b t) d'))
    z_pred = rearrange(z_pred, '(b t) d -> b t d', b=B)

    # 3. Prediction loss
    pred_loss = F.mse_loss(z_pred, z_target)

    # 4. Anti-collapse regularizer on the full embedding distribution
    reg = covariance_regularizer(rearrange(z, 'b t d -> (b t) d'))

    total = pred_loss + lam * reg
    return total, {'pred': pred_loss.item(), 'reg': reg.item(), 'total': total.item()}


# One forward pass to verify everything wires up
encoder.train(); predictor.train()
test_batch = next(iter(loader)).to(device)
loss, components = lewm_loss(test_batch, encoder, predictor)
print('First forward pass:')
for k, v in components.items():
    print(f'  {k}: {v:.4f}')

## 7. Run a real training loop

Now we put it all together. A few epochs on synthetic data — should take ~30-60 seconds on a T4 GPU.

Watch the loss curve. If `pred` decreases steadily and `reg` stays stable, training is working. If `pred` immediately drops to ~0 and `reg` shoots up, you have collapse — bump up `lam`.

In [ ]:
encoder = TinyEncoder(embed_dim=64).to(device)
predictor = TinyPredictor(embed_dim=64).to(device)
optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=1e-3, weight_decay=1e-4,
)

n_epochs = 10
history = {'pred': [], 'reg': [], 'total': []}

for epoch in range(n_epochs):
    encoder.train(); predictor.train()
    epoch_losses = {'pred': [], 'reg': [], 'total': []}

    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
    for clips in pbar:
        clips = clips.to(device)
        loss, components = lewm_loss(clips, encoder, predictor, lam=0.5)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k, v in components.items():
            epoch_losses[k].append(v)
        pbar.set_postfix({k: f'{v:.3f}' for k, v in components.items()})

    for k in history:
        history[k].append(np.mean(epoch_losses[k]))
    print(f'Epoch {epoch+1:2d}: pred={history["pred"][-1]:.4f}  reg={history["reg"][-1]:.4f}  total={history["total"][-1]:.4f}')

In [ ]:
# Plot the loss curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, color in zip(axes, ['pred', 'reg', 'total'], ['#c0ff3e', '#ff3e7e', '#3eff9e']):
    ax.plot(history[key], color=color, linewidth=2)
    ax.set_title(f'{key} loss')
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
plt.suptitle('Training progress', y=1.02)
plt.tight_layout()
plt.show()

## 8. Test the surprise mechanism

This is the core of your deepfake detection idea. Generate two clips — one normal, one with a teleportation — and compute per-frame surprise.

If our trained model has learned anything, the perturbed clip should have a visible spike at the teleport frame.

In [ ]:
@torch.no_grad()
def compute_surprise(clip_np, encoder, predictor):
    """Per-frame surprise scores for a single video clip."""
    encoder.eval(); predictor.eval()
    clip = torch.from_numpy(clip_np).float() / 255.0
    clip = rearrange(clip, 't h w c -> t c h w').to(device)
    z = encoder(clip)                   # (T, D)
    z_pred = predictor(z[:-1])         # (T-1, D) predicting frames 1..T-1
    surprise = ((z_pred - z[1:]) ** 2).mean(dim=-1)  # (T-1,)
    return surprise.cpu().numpy()


# Generate one clean clip and one with a teleportation
clean_clip = make_synthetic_clip(n_frames=24, size=64, perturbed=False)
perturbed_clip = make_synthetic_clip(n_frames=24, size=64, perturbed=True)

clean_surprise = compute_surprise(clean_clip, encoder, predictor)
perturbed_surprise = compute_surprise(perturbed_clip, encoder, predictor)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(clean_surprise, label='Clean (no teleport)', color='#3eff9e', linewidth=2)
ax.plot(perturbed_surprise, label='Perturbed (teleport at frame 12)', color='#ff3e7e', linewidth=2)
ax.axvline(11, color='#ffb13e', linestyle='--', alpha=0.5, label='Teleport injected here')
ax.set_xlabel('Frame index')
ax.set_ylabel('Surprise score (MSE)')
ax.set_title('Per-frame surprise: clean vs teleport-perturbed')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nMean surprise — clean:     {clean_surprise.mean():.4f}')
print(f'Mean surprise — perturbed: {perturbed_surprise.mean():.4f}')
print(f'Spike at teleport frame:   {perturbed_surprise[11]:.4f} (vs neighbors: {perturbed_surprise[10]:.4f}, {perturbed_surprise[12]:.4f})')

if perturbed_surprise[11] > perturbed_surprise.mean() * 1.3:
    print('\n✅ Surprise mechanism works! The model detects physical violations.')
else:
    print('\n⚠️  No clear spike yet. Train for more epochs or larger dataset.')

In [ ]:
# Fair comparison: same clip dynamics, only difference is the teleport
def make_paired_clips(n_frames=24, size=64, seed=42):
    """Generate a clean and perturbed version of the SAME underlying clip."""
    np.random.seed(seed)
    clean = make_synthetic_clip(n_frames, size, perturbed=False)
    np.random.seed(seed)  # reset so we get same starting conditions
    perturbed = make_synthetic_clip(n_frames, size, perturbed=True)
    return clean, perturbed

# Try a few seeds and average
results_clean = []
results_perturbed = []
for seed in range(20):
    clean, perturbed = make_paired_clips(seed=seed)
    s_clean = compute_surprise(clean, encoder, predictor)
    s_pert = compute_surprise(perturbed, encoder, predictor)
    results_clean.append(s_clean)
    results_perturbed.append(s_pert)

# Average across the 20 trials
mean_clean = np.mean(results_clean, axis=0)
mean_perturbed = np.mean(results_perturbed, axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mean_clean, label='Clean (avg of 20 seeds)', color='#3eff9e', linewidth=2)
ax.plot(mean_perturbed, label='Perturbed (avg of 20 seeds)', color='#ff3e7e', linewidth=2)
ax.axvline(11, color='#ffb13e', linestyle='--', alpha=0.5, label='Teleport here')
ax.set_xlabel('Frame index')
ax.set_ylabel('Mean surprise (over 20 trials)')
ax.set_title('Per-frame surprise: paired clean vs perturbed (same underlying clips)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Specifically check the spike
print(f"\nMean surprise at frame 11 (teleport):")
print(f"  Clean:     {mean_clean[11]:.5f}")
print(f"  Perturbed: {mean_perturbed[11]:.5f}")
print(f"  Difference: {(mean_perturbed[11] - mean_clean[11]):.5f}")

if mean_perturbed[11] > mean_clean[11] * 1.1:
    print("✅ Surprise mechanism works! Teleport detection confirmed.")
else:
    print("⚠️ Still not detecting cleanly. Try training more epochs.")

## 9. Save the checkpoint

Optionally save the trained model so you can use it in the next session. Mount Google Drive first if you want it to persist beyond this Colab session.

In [ ]:
checkpoint = {
    'encoder_state': encoder.state_dict(),
    'predictor_state': predictor.state_dict(),
    'history': history,
    'config': {
        'embed_dim': 128,
        'image_size': 64,
    }
}
torch.save(checkpoint, 'lewm_starter_checkpoint.pt')
print('✓ Saved to lewm_starter_checkpoint.pt')
print('  (download via the Files panel on the left, or mount Drive to persist)')

In [ ]:
# ============================================================
# DAY 3 — Real video training (HuggingFace mirror)
# ============================================================

import os
import shutil
from pathlib import Path

DATA_DIR = Path('/content/hmdb51')
DATA_DIR.mkdir(exist_ok=True)
extracted_marker = DATA_DIR / '.extracted'

# Clean up any partial download from before
old_archive = DATA_DIR / 'hmdb51_org.rar'
if old_archive.exists():
    print('Removing partial download from previous attempt...')
    old_archive.unlink()

if extracted_marker.exists():
    print('✓ HMDB51 already prepared')
else:
    print('Installing huggingface_hub...')
    !pip install -q huggingface_hub

    print('\nDownloading HMDB51 from HuggingFace mirror (~2GB)...')
    print('This usually takes 3-7 min on Colab.\n')

    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="jili5044/hmdb51",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        local_dir_use_symlinks=False,
    )

    extracted_marker.touch()
    print('\n✓ Download complete!')

# Show what we got
print(f'\nContents of {DATA_DIR}:')
def walk_summary(path, depth=0, max_depth=2):
    for item in sorted(path.iterdir())[:10]:
        indent = '  ' * depth
        if item.is_dir():
            n_files = sum(1 for _ in item.iterdir())
            print(f'{indent}{item.name}/ ({n_files} items)')
            if depth < max_depth:
                walk_summary(item, depth+1, max_depth)
        else:
            size_mb = item.stat().st_size / 1e6
            print(f'{indent}{item.name} ({size_mb:.1f} MB)')

walk_summary(DATA_DIR)

In [ ]:
import zipfile
from pathlib import Path

DATA_DIR = Path('/content/hmdb51')
zip_path = DATA_DIR / 'hmdb51.zip'
extract_dir = DATA_DIR / 'extracted'

if extract_dir.exists() and any(extract_dir.iterdir()):
    print(f'✓ Already extracted at {extract_dir}')
else:
    print('Extracting hmdb51.zip ...')
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print('✓ Extraction complete')

# Show what we got
print(f'\nTop-level contents of {extract_dir}:')
items = sorted(extract_dir.iterdir())
for item in items[:15]:
    if item.is_dir():
        n = sum(1 for _ in item.iterdir())
        print(f'  📁 {item.name}/  ({n} items inside)')
    else:
        print(f'  📄 {item.name}')
if len(items) > 15:
    print(f'  ... and {len(items) - 15} more')

# Find any .avi files
print(f'\nSearching for .avi files (sample of 5):')
avi_files = list(extract_dir.rglob('*.avi'))
print(f'  Total .avi files found: {len(avi_files)}')
for f in avi_files[:5]:
    rel = f.relative_to(extract_dir)
    size_mb = f.stat().st_size / 1e6
    print(f'  • {rel}  ({size_mb:.2f} MB)')

In [ ]:
import cv2
import numpy as np
from pathlib import Path

VIDEOS_ROOT = Path('/content/hmdb51/extracted/hmdb51')

# Pick the first video we find and try to read it
all_videos = list(VIDEOS_ROOT.rglob('*.avi'))
print(f'Total videos: {len(all_videos)}')

# Try reading the first one
test_path = all_videos[0]
print(f'\nTesting read on: {test_path.name}')

cap = cv2.VideoCapture(str(test_path))
if not cap.isOpened():
    print('❌ Could not open video — codec issue. Tell me!')
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'  ✓ Opens correctly')
    print(f'  Frames: {n_frames}')
    print(f'  Resolution: {width}×{height}')
    print(f'  FPS: {fps:.1f}')
    print(f'  Duration: {n_frames/fps:.2f}s')

    # Try reading a frame
    ret, frame = cap.read()
    if ret:
        print(f'  ✓ First frame read: shape {frame.shape}, dtype {frame.dtype}')
    else:
        print('  ❌ Could not read first frame')
    cap.release()

# Check stats across many videos so we know what to expect
print(f'\nSurveying 50 random videos for size/length stats...')
import random
random.seed(42)
sample = random.sample(all_videos, min(50, len(all_videos)))

durations, widths, heights = [], [], []
for v in sample:
    cap = cv2.VideoCapture(str(v))
    if cap.isOpened():
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        durations.append(n / fps)
        widths.append(int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
        heights.append(int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()

print(f'  Frame counts:  min={min(int(d*30) for d in durations)},  median≈{int(sorted([d*30 for d in durations])[len(durations)//2])},  max={max(int(d*30) for d in durations)}')
print(f'  Widths:        unique values = {sorted(set(widths))}')
print(f'  Heights:       unique values = {sorted(set(heights))}')
print(f'  Durations:     {min(durations):.1f}s to {max(durations):.1f}s')

# Visual sanity check: show first frame of 4 different videos
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, v in zip(axes, random.sample(all_videos, 4)):
    cap = cv2.VideoCapture(str(v))
    ret, frame = cap.read()
    cap.release()
    if ret:
        # OpenCV reads BGR, convert to RGB for display
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
    action = v.parent.name
    ax.set_title(f'{action}\n{v.name[:30]}...', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample frames from HMDB51', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path
from einops import rearrange
import random

class HMDB51Dataset(Dataset):
    """
    Loads random 16-frame clips from HMDB51 videos.

    Returns clips in the same shape as SyntheticVideoDataset:
        (T, C, H, W) float tensor in [0, 1], RGB.

    Drop-in replacement — the rest of the training pipeline is unchanged.
    """

    def __init__(self, video_paths, n_frames=16, size=64, min_video_frames=20):
        # Pre-filter: only keep videos with enough frames
        self.video_paths = []
        for p in video_paths:
            cap = cv2.VideoCapture(str(p))
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            if n >= min_video_frames:
                self.video_paths.append(p)
        self.n_frames = n_frames
        self.size = size
        print(f'  Dataset ready: {len(self.video_paths)} usable videos')

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        cap = cv2.VideoCapture(str(path))
        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Pick a random starting offset so each epoch sees different chunks
        max_start = max(0, n_total - self.n_frames)
        start = random.randint(0, max_start)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

        frames = []
        for _ in range(self.n_frames):
            ret, frame = cap.read()
            if not ret:
                # Hit end of video — repeat last frame as padding
                if frames:
                    frames.append(frames[-1])
                else:
                    # Couldn't read at all — fallback to zeros
                    frames.append(np.zeros((self.size, self.size, 3), dtype=np.uint8))
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = self._center_crop_resize(frame, self.size)
            frames.append(frame)

        cap.release()

        # Stack and convert to tensor (T, C, H, W) in [0, 1]
        clip = np.stack(frames, axis=0)
        clip = torch.from_numpy(clip).float() / 255.0
        clip = rearrange(clip, 't h w c -> t c h w')
        return clip

    @staticmethod
    def _center_crop_resize(img, size):
        """Center-crop to a square then resize to (size, size)."""
        h, w = img.shape[:2]
        s = min(h, w)
        y0 = (h - s) // 2
        x0 = (w - s) // 2
        img = img[y0:y0+s, x0:x0+s]
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
        return img


# Build the dataset (this takes ~30-60s because we pre-filter)
print('Building HMDB51 dataset (pre-filtering by frame count, ~30-60s)...')
all_videos = list(Path('/content/hmdb51/extracted/hmdb51').rglob('*.avi'))
print(f'  Found {len(all_videos)} .avi files')

# Use 1000 videos for speed (full 6754 would be slow per epoch)
random.seed(42)
random.shuffle(all_videos)
subset = all_videos[:1000]

real_dataset = HMDB51Dataset(subset, n_frames=16, size=64)
real_loader = DataLoader(real_dataset, batch_size=16, num_workers=2, shuffle=True)

# Sanity check: grab one batch
print('\nLoading a sample batch (~5-10s)...')
batch = next(iter(real_loader))
print(f'Batch shape: {batch.shape}')
print(f'Value range: [{batch.min():.3f}, {batch.max():.3f}]')

# Visualize the sample so you can see what the model will train on
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
clip_to_show = batch[0]  # first clip
clip_to_show2 = batch[1]  # second clip
for i in range(8):
    img1 = clip_to_show[i*2].permute(1, 2, 0).numpy()
    img2 = clip_to_show2[i*2].permute(1, 2, 0).numpy()
    axes[0, i].imshow(img1)
    axes[0, i].set_title(f't={i*2}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(img2)
    axes[1, i].set_title(f't={i*2}', fontsize=9)
    axes[1, i].axis('off')
plt.suptitle('Two real video clips (every 2nd frame), 64×64 RGB', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import time
import torch
import numpy as np
from tqdm.notebook import tqdm

# Re-instantiate fresh model (don't reuse the synthetic-trained one)
encoder_real = TinyEncoder(embed_dim=64).to(device)
predictor_real = TinyPredictor(embed_dim=64).to(device)
optimizer = torch.optim.AdamW(
    list(encoder_real.parameters()) + list(predictor_real.parameters()),
    lr=1e-3, weight_decay=1e-4,
)

n_epochs = 8
history_real = {'pred': [], 'reg': [], 'total': []}

print(f'Training on real video — {n_epochs} epochs, ~1000 clips per epoch')
print(f'Expect ~3-6 min per epoch on T4 GPU.\n')
t_start = time.time()

for epoch in range(n_epochs):
    encoder_real.train(); predictor_real.train()
    epoch_losses = {'pred': [], 'reg': [], 'total': []}

    pbar = tqdm(real_loader, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
    for clips in pbar:
        clips = clips.to(device)
        loss, components = lewm_loss(clips, encoder_real, predictor_real, lam=0.5)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k, v in components.items():
            epoch_losses[k].append(v)
        pbar.set_postfix({k: f'{v:.3f}' for k, v in components.items()})

    for k in history_real:
        history_real[k].append(np.mean(epoch_losses[k]))

    elapsed = time.time() - t_start
    print(f'Epoch {epoch+1:2d}: pred={history_real["pred"][-1]:.4f}  '
          f'reg={history_real["reg"][-1]:.4f}  total={history_real["total"][-1]:.4f}  '
          f'[{elapsed/60:.1f} min elapsed]')

print(f'\n✓ Training done in {(time.time()-t_start)/60:.1f} min')

# Plot the loss curves
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, color in zip(axes, ['pred', 'reg', 'total'], ['#c0ff3e', '#ff3e7e', '#3eff9e']):
    ax.plot(history_real[key], color=color, linewidth=2, marker='o')
    ax.set_title(f'{key} loss (real video)')
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
plt.suptitle('Training on real video (HMDB51)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print(f'Total batches per epoch: {len(real_loader)}')
print(f'Expected batches: {len(real_dataset) // 16}')

# Time how long iterating once actually takes
import time
t = time.time()
for i, batch in enumerate(real_loader):
    if i == 0:
        print(f'  First batch: shape {batch.shape}')
print(f'Full pass: {time.time() - t:.1f}s, {i+1} batches')

In [ ]:
# Diagnostic: does the trained model produce DIFFERENT surprise scores
# than a random untrained model? If not, training didn't help.

import numpy as np
import torch
from einops import rearrange
import matplotlib.pyplot as plt

@torch.no_grad()
def surprise_for_clip(clip_tensor, enc, pred):
    """Compute per-frame surprise for a clip already in tensor form."""
    enc.eval(); pred.eval()
    z = enc(clip_tensor.to(device))
    z_pred = pred(z[:-1])
    s = ((z_pred - z[1:]) ** 2).mean(dim=-1)
    return s.cpu().numpy()


# Build a fresh, UNTRAINED model
encoder_random = TinyEncoder(embed_dim=64).to(device)
predictor_random = TinyPredictor(embed_dim=64).to(device)

# Grab 5 real clips
sample_batch = next(iter(real_loader))[:5]  # (5, 16, 3, 64, 64)

print('Comparing surprise scores: trained vs untrained on the same clips\n')
print(f'{"Clip":<6} {"Trained mean":>14} {"Untrained mean":>16} {"Same?":>8}')
print('-' * 50)

for i in range(5):
    clip = sample_batch[i]  # (16, 3, 64, 64)

    s_trained = surprise_for_clip(clip, encoder_real, predictor_real)
    s_random = surprise_for_clip(clip, encoder_random, predictor_random)

    same = abs(s_trained.mean() - s_random.mean()) < 0.001
    print(f'{i:<6} {s_trained.mean():>14.5f} {s_random.mean():>16.5f} {"✅" if not same else "⚠️"}')

# Also check: are the trained embeddings actually using the latent space?
print('\nLatent space usage (should be > 0.1 for good models):')
with torch.no_grad():
    encoder_real.eval()
    flat = rearrange(sample_batch.to(device), 'b t c h w -> (b t) c h w')
    z = encoder_real(flat)
    print(f'  Embedding std (per dim): {z.std(dim=0).mean():.4f}')
    print(f'  Embedding range: [{z.min():.3f}, {z.max():.3f}]')
    print(f'  Active dimensions (std > 0.1): {(z.std(dim=0) > 0.1).sum().item()} / {z.shape[1]}')

In [ ]:
import torch
import numpy as np
from einops import rearrange
import matplotlib.pyplot as plt

@torch.no_grad()
def perturb_clip(clip, perturb_frame=8):
    """
    Take a clip and inject a 'teleport' at perturb_frame by replacing it
    with a random frame from a totally different clip.
    Mimics the kind of temporal discontinuity AI-generated video has.
    """
    clip = clip.clone()  # (T, C, H, W)
    # Get a random frame from a totally different clip
    other = next(iter(real_loader))[0]  # different random clip from dataloader
    other_frame = other[5]  # arbitrary frame from it
    clip[perturb_frame] = other_frame
    return clip


def per_frame_surprise(clip_tensor, enc, pred):
    """Compute surprise for each frame transition."""
    enc.eval(); pred.eval()
    with torch.no_grad():
        z = enc(clip_tensor.to(device))            # (T, D)
        z_pred = pred(z[:-1])                       # predict z_1..z_{T-1}
        s = ((z_pred - z[1:]) ** 2).mean(dim=-1)   # (T-1,)
    return s.cpu().numpy()


# Average over 30 different real clips for a robust signal
N_TRIALS = 30
PERTURB_FRAME = 8

print(f'Running {N_TRIALS} paired trials (clean vs perturbed)...')

clean_results = []
perturbed_results = []

# Get 30 different clips by iterating the dataloader
clips_seen = 0
for batch in real_loader:
    for i in range(batch.shape[0]):
        if clips_seen >= N_TRIALS:
            break
        clip = batch[i]  # (16, 3, 64, 64)
        perturbed = perturb_clip(clip, perturb_frame=PERTURB_FRAME)

        s_clean = per_frame_surprise(clip, encoder_real, predictor_real)
        s_pert = per_frame_surprise(perturbed, encoder_real, predictor_real)

        clean_results.append(s_clean)
        perturbed_results.append(s_pert)
        clips_seen += 1
    if clips_seen >= N_TRIALS:
        break

mean_clean = np.mean(clean_results, axis=0)
mean_perturbed = np.mean(perturbed_results, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mean_clean, label=f'Clean (avg of {N_TRIALS})',
        color='#3eff9e', linewidth=2, marker='o')
ax.plot(mean_perturbed, label=f'Perturbed (frame swap at {PERTURB_FRAME})',
        color='#ff3e7e', linewidth=2, marker='o')
ax.axvline(PERTURB_FRAME-1, color='#ffb13e', linestyle='--', alpha=0.7,
           label=f'Perturbation injected here')
ax.set_xlabel('Frame transition index (predicting frame N+1 from N)')
ax.set_ylabel('Mean surprise (MSE)')
ax.set_title('Real video: clean vs perturbed (HMDB51 trained model)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Stats
print(f'\n=== Surprise around the perturbation ===')
print(f'Frame {PERTURB_FRAME-1} (just before):')
print(f'  Clean:     {mean_clean[PERTURB_FRAME-1]:.5f}')
print(f'  Perturbed: {mean_perturbed[PERTURB_FRAME-1]:.5f}')
print(f'  Ratio:     {mean_perturbed[PERTURB_FRAME-1] / mean_clean[PERTURB_FRAME-1]:.2f}x')

if mean_perturbed[PERTURB_FRAME-1] > mean_clean[PERTURB_FRAME-1] * 1.3:
    print('\n✅ STRONG signal — the model detects the perturbation')
elif mean_perturbed[PERTURB_FRAME-1] > mean_clean[PERTURB_FRAME-1] * 1.1:
    print('\n🟡 Weak but real signal — detection works at this level')
else:
    print('\n⚠️  No clear signal — perturbation type might be too subtle for this model')

In [ ]:
import torch

# Save the real-video-trained checkpoint
checkpoint_real = {
    'encoder_state': encoder_real.state_dict(),
    'predictor_state': predictor_real.state_dict(),
    'history': history_real,
    'config': {
        'embed_dim': 64,
        'image_size': 64,
        'n_frames': 16,
        'lam': 0.5,
        'n_epochs': 8,
        'dataset': 'HMDB51 (1000 clip subset)',
        'training_time_min': 2.6,
    }
}
torch.save(checkpoint_real, 'lewm_hmdb51_checkpoint.pt')
print('✓ Saved to lewm_hmdb51_checkpoint.pt')

# Verify file size
from pathlib import Path
size_mb = Path('lewm_hmdb51_checkpoint.pt').stat().st_size / 1e6
print(f'  File size: {size_mb:.2f} MB')
print('\n📥 Download this file via the Files panel on the left:')
print('   1. Click the folder icon on the left sidebar')
print('   2. Find "lewm_hmdb51_checkpoint.pt"')
print('   3. Right-click → Download')
print('\nThis is the model we\'ll integrate into your live demo backend.')

In [ ]:
import time
import random
import torch
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Build a bigger dataset (5000 clips this time, takes ~2-3 min to filter)
print('Building larger HMDB51 dataset (5000 clips)...')
all_videos = list(Path('/content/hmdb51/extracted/hmdb51').rglob('*.avi'))
random.seed(42)
random.shuffle(all_videos)
subset_5k = all_videos[:5000]

real_dataset_big = HMDB51Dataset(subset_5k, n_frames=16, size=64)
real_loader_big = DataLoader(real_dataset_big, batch_size=16, num_workers=2, shuffle=True)
print(f'  ✓ {len(real_dataset_big)} usable clips, {len(real_loader_big)} batches per epoch')

# Fresh model
encoder_big = TinyEncoder(embed_dim=64).to(device)
predictor_big = TinyPredictor(embed_dim=64).to(device)
optimizer = torch.optim.AdamW(
    list(encoder_big.parameters()) + list(predictor_big.parameters()),
    lr=1e-3, weight_decay=1e-4,
)

n_epochs = 12
history_big = {'pred': [], 'reg': [], 'total': []}

print(f'\nTraining for {n_epochs} epochs on {len(real_dataset_big)} clips...')
print('Expect ~10-15 min total based on previous timing.\n')
t_start = time.time()

for epoch in range(n_epochs):
    encoder_big.train(); predictor_big.train()
    epoch_losses = {'pred': [], 'reg': [], 'total': []}

    pbar = tqdm(real_loader_big, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
    for clips in pbar:
        clips = clips.to(device)
        loss, components = lewm_loss(clips, encoder_big, predictor_big, lam=0.5)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k, v in components.items():
            epoch_losses[k].append(v)
        pbar.set_postfix({k: f'{v:.3f}' for k, v in components.items()})

    for k in history_big:
        history_big[k].append(np.mean(epoch_losses[k]))

    elapsed = time.time() - t_start
    print(f'Epoch {epoch+1:2d}: pred={history_big["pred"][-1]:.4f}  '
          f'reg={history_big["reg"][-1]:.4f}  total={history_big["total"][-1]:.4f}  '
          f'[{elapsed/60:.1f} min elapsed]')

print(f'\n✓ Training done in {(time.time()-t_start)/60:.1f} min')

# Plot losses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, color in zip(axes, ['pred', 'reg', 'total'], ['#c0ff3e', '#ff3e7e', '#3eff9e']):
    ax.plot(history_big[key], color=color, linewidth=2, marker='o')
    ax.set_title(f'{key} loss (5000 clips, 12 epochs)')
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
plt.suptitle('Training on real video — bigger run', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
from einops import rearrange
import matplotlib.pyplot as plt

# Run 50 paired trials this time for tighter signal
N_TRIALS = 50
PERTURB_FRAME = 8

print(f'Running {N_TRIALS} paired trials on the big-run model...\n')

clean_results = []
perturbed_results = []

clips_seen = 0
for batch in real_loader_big:
    for i in range(batch.shape[0]):
        if clips_seen >= N_TRIALS:
            break
        clip = batch[i]
        perturbed = perturb_clip(clip, perturb_frame=PERTURB_FRAME)

        s_clean = per_frame_surprise(clip, encoder_big, predictor_big)
        s_pert = per_frame_surprise(perturbed, encoder_big, predictor_big)

        clean_results.append(s_clean)
        perturbed_results.append(s_pert)
        clips_seen += 1
    if clips_seen >= N_TRIALS:
        break

mean_clean_big = np.mean(clean_results, axis=0)
mean_perturbed_big = np.mean(perturbed_results, axis=0)

# Side-by-side comparison: small run vs big run
fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)

# Small run (the previous mean_clean / mean_perturbed)
axes[0].plot(mean_clean, label=f'Clean', color='#3eff9e', linewidth=2, marker='o')
axes[0].plot(mean_perturbed, label=f'Perturbed', color='#ff3e7e', linewidth=2, marker='o')
axes[0].axvline(PERTURB_FRAME-1, color='#ffb13e', linestyle='--', alpha=0.7)
axes[0].set_title('Small run: 1000 clips × 8 epochs')
axes[0].set_xlabel('Frame transition')
axes[0].set_ylabel('Mean surprise')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Big run
axes[1].plot(mean_clean_big, label=f'Clean', color='#3eff9e', linewidth=2, marker='o')
axes[1].plot(mean_perturbed_big, label=f'Perturbed', color='#ff3e7e', linewidth=2, marker='o')
axes[1].axvline(PERTURB_FRAME-1, color='#ffb13e', linestyle='--', alpha=0.7)
axes[1].set_title('Big run: 5000 clips × 12 epochs')
axes[1].set_xlabel('Frame transition')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Quantitative comparison
small_ratio = mean_perturbed[PERTURB_FRAME-1] / mean_clean[PERTURB_FRAME-1]
big_ratio = mean_perturbed_big[PERTURB_FRAME-1] / mean_clean_big[PERTURB_FRAME-1]

print('=== Spike magnitude comparison ===')
print(f'Small run (1000×8):  {small_ratio:.2f}× clean baseline')
print(f'Big run (5000×12):   {big_ratio:.2f}× clean baseline')
print(f'Improvement: {(big_ratio - small_ratio):.2f}× absolute, {((big_ratio/small_ratio - 1)*100):+.0f}% relative')

if big_ratio > small_ratio * 1.1:
    print('\n✅ Bigger model gives a stronger spike — scaling helps.')
elif big_ratio > small_ratio * 0.9:
    print('\n🟡 Roughly equivalent — model is at its capacity ceiling.')
else:
    print('\n⚠️  Big run is actually weaker. Likely overtraining (pred loss climbed).')

In [ ]:
import time
import random
import copy
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# ============================================================
# 1. Build train/val split
# ============================================================
print('Building train/val split...')
all_videos = list(Path('/content/hmdb51/extracted/hmdb51').rglob('*.avi'))
random.seed(42)
random.shuffle(all_videos)

# 5000 train + 300 val (held out, never trained on)
train_videos = all_videos[:5000]
val_videos = all_videos[5000:5300]

train_dataset = HMDB51Dataset(train_videos, n_frames=16, size=64)
val_dataset = HMDB51Dataset(val_videos, n_frames=16, size=64)

train_loader = DataLoader(train_dataset, batch_size=16, num_workers=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=2, shuffle=False)
print(f'  Train: {len(train_dataset)} clips, {len(train_loader)} batches')
print(f'  Val:   {len(val_dataset)} clips, {len(val_loader)} batches')

# ============================================================
# 2. Validation function — computes the spike ratio on held-out data
# ============================================================
@torch.no_grad()
def evaluate_spike_ratio(enc, pred, loader, n_trials=20, perturb_frame=8):
    """
    Returns the perturbed/clean surprise ratio at the perturbation frame,
    averaged over n_trials clip pairs from the val loader.
    Higher = better detection.
    """
    enc.eval(); pred.eval()
    clean_surprises = []
    pert_surprises = []
    trials = 0

    val_iter = iter(loader)
    pert_source = next(val_iter)  # source of "swap-in" frames

    while trials < n_trials:
        try:
            batch = next(val_iter)
        except StopIteration:
            break
        for i in range(batch.shape[0]):
            if trials >= n_trials:
                break
            clip = batch[i].to(device)
            pert = clip.clone()
            pert[perturb_frame] = pert_source[(trials * 3) % pert_source.shape[0], 5].to(device)

            # surprise per frame transition
            z_c = enc(clip); z_p = enc(pert)
            zhat_c = pred(z_c[:-1]); zhat_p = pred(z_p[:-1])
            s_c = ((zhat_c - z_c[1:]) ** 2).mean(dim=-1).cpu().numpy()
            s_p = ((zhat_p - z_p[1:]) ** 2).mean(dim=-1).cpu().numpy()

            clean_surprises.append(s_c)
            pert_surprises.append(s_p)
            trials += 1

    mc = np.mean(clean_surprises, axis=0)
    mp = np.mean(pert_surprises, axis=0)
    ratio = mp[perturb_frame - 1] / max(mc[perturb_frame - 1], 1e-9)
    return float(ratio), mc, mp

# ============================================================
# 3. Training loop with validation + checkpoint saving
# ============================================================
encoder_es = TinyEncoder(embed_dim=64).to(device)
predictor_es = TinyPredictor(embed_dim=64).to(device)
optimizer = torch.optim.AdamW(
    list(encoder_es.parameters()) + list(predictor_es.parameters()),
    lr=1e-3, weight_decay=1e-4,
)

n_epochs = 15
history_es = {'pred': [], 'reg': [], 'total': [], 'val_spike': []}

# Track the best model seen so far
best_spike = 0.0
best_epoch = -1
best_encoder_state = None
best_predictor_state = None

print(f'\nTraining {n_epochs} epochs with early stopping based on val spike ratio...\n')
t_start = time.time()

for epoch in range(n_epochs):
    # ---- Train ----
    encoder_es.train(); predictor_es.train()
    epoch_losses = {'pred': [], 'reg': [], 'total': []}
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{n_epochs} (train)', leave=False)
    for clips in pbar:
        clips = clips.to(device)
        loss, components = lewm_loss(clips, encoder_es, predictor_es, lam=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        for k, v in components.items():
            epoch_losses[k].append(v)
    for k in ['pred', 'reg', 'total']:
        history_es[k].append(np.mean(epoch_losses[k]))

    # ---- Validate ----
    val_spike, _, _ = evaluate_spike_ratio(encoder_es, predictor_es, val_loader, n_trials=20)
    history_es['val_spike'].append(val_spike)

    # ---- Save if best so far ----
    is_best = val_spike > best_spike
    if is_best:
        best_spike = val_spike
        best_epoch = epoch + 1
        best_encoder_state = copy.deepcopy(encoder_es.state_dict())
        best_predictor_state = copy.deepcopy(predictor_es.state_dict())

    elapsed = time.time() - t_start
    marker = '⭐ NEW BEST' if is_best else ''
    print(f'Epoch {epoch+1:2d}: pred={history_es["pred"][-1]:.4f}  '
          f'reg={history_es["reg"][-1]:.4f}  '
          f'val_spike={val_spike:.2f}×  '
          f'[{elapsed/60:.1f} min]  {marker}')

print(f'\n✓ Training done in {(time.time()-t_start)/60:.1f} min')
print(f'\n🏆 Best epoch: {best_epoch} with spike ratio {best_spike:.2f}×')
print(f'   (final epoch had spike ratio {history_es["val_spike"][-1]:.2f}×)')

# ============================================================
# 4. Restore best checkpoint
# ============================================================
encoder_es.load_state_dict(best_encoder_state)
predictor_es.load_state_dict(best_predictor_state)
print('✓ Restored best-epoch weights')

# ============================================================
# 5. Plot training curves with val spike overlaid
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(history_es['pred'], color='#c0ff3e', linewidth=2, marker='o')
axes[0].set_title('pred loss')
axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(history_es['reg'], color='#ff3e7e', linewidth=2, marker='o')
axes[1].set_title('reg loss')
axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(history_es['total'], color='#3eff9e', linewidth=2, marker='o')
axes[2].set_title('total loss')
axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

axes[3].plot(history_es['val_spike'], color='#ffb13e', linewidth=2, marker='o')
axes[3].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[3].set_title('val spike ratio (★ this is what we optimize)')
axes[3].set_xlabel('Epoch')
axes[3].set_ylabel('Perturbed / clean ratio')
axes[3].legend()
axes[3].grid(alpha=0.3)

plt.suptitle('Training with validation-based early stopping', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import torch
from pathlib import Path

# Save the early-stopped model
checkpoint_es = {
    'encoder_state': encoder_es.state_dict(),
    'predictor_state': predictor_es.state_dict(),
    'history': history_es,
    'best_epoch': best_epoch,
    'best_spike_ratio': best_spike,
    'config': {
        'embed_dim': 64,
        'image_size': 64,
        'n_frames': 16,
        'lam': 0.5,
        'n_epochs_trained': 15,
        'best_epoch': best_epoch,
        'training_data': 'HMDB51 (5000 train clips)',
        'val_data': 'HMDB51 (300 held-out clips)',
        'val_metric': 'spike_ratio',
        'val_metric_value': best_spike,
    }
}
torch.save(checkpoint_es, 'lewm_hmdb51_earlystop.pt')
size_mb = Path('lewm_hmdb51_earlystop.pt').stat().st_size / 1e6
print(f'✓ Saved to lewm_hmdb51_earlystop.pt ({size_mb:.2f} MB)')
print(f'  Best epoch: {best_epoch}')
print(f'  Best spike ratio: {best_spike:.2f}×')
print(f'\n⬇️  Download this file via the Files panel on the left:')
print(f'   1. Click the folder icon (sidebar)')
print(f'   2. Find lewm_hmdb51_earlystop.pt')
print(f'   3. Right-click → Download')
print(f'\nThis is the model you\'ll integrate into your live demo.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from einops import rearrange

# ============================================================
# Run all 4 trained models on the same paired test
# (so the comparison is fair — same clips, same perturbation)
# ============================================================

@torch.no_grad()
def measure_model(enc, pred, loader, n_trials=30, perturb_frame=8):
    enc.eval(); pred.eval()
    cleans, perts = [], []
    pert_source_batch = next(iter(loader))
    val_iter = iter(loader)
    trials = 0
    while trials < n_trials:
        try:
            batch = next(val_iter)
        except StopIteration:
            break
        for i in range(batch.shape[0]):
            if trials >= n_trials: break
            clip = batch[i].to(device)
            pert = clip.clone()
            pert[perturb_frame] = pert_source_batch[(trials*7) % pert_source_batch.shape[0], 5].to(device)
            z_c, z_p = enc(clip), enc(pert)
            zhat_c, zhat_p = pred(z_c[:-1]), pred(z_p[:-1])
            s_c = ((zhat_c - z_c[1:])**2).mean(dim=-1).cpu().numpy()
            s_p = ((zhat_p - z_p[1:])**2).mean(dim=-1).cpu().numpy()
            cleans.append(s_c); perts.append(s_p)
            trials += 1
    return np.mean(cleans, axis=0), np.mean(perts, axis=0)

print('Measuring all 4 models on the val set (~1 min)...')

# Use val_loader so the comparison is on data NONE of the models trained on
mc1, mp1 = measure_model(encoder, predictor, val_loader)            # synthetic-trained
mc2, mp2 = measure_model(encoder_real, predictor_real, val_loader)  # small run
mc3, mp3 = measure_model(encoder_big, predictor_big, val_loader)    # big run (overtrained)
mc4, mp4 = measure_model(encoder_es, predictor_es, val_loader)      # early-stopped (best)

# ============================================================
# The big plot — 4-panel grid telling the whole story
# ============================================================
fig = plt.figure(figsize=(16, 9), facecolor='#0a0a0f')
fig.suptitle('SURPRISE — Day 3: Training a deepfake detector from scratch',
             color='#c0ff3e', fontsize=18, fontweight='bold', y=0.98)

models = [
    ('1. Synthetic only', mc1, mp1, 'Trained on bouncing squares.\nNever saw real video.', '#888'),
    ('2. Small real run',  mc2, mp2, '1000 HMDB51 clips, 8 epochs.\nFirst real-video model.', '#3eff9e'),
    ('3. Big run (overtrained)', mc3, mp3, '5000 clips, 12 epochs, no validation.\nClassic overtraining.', '#ff3e7e'),
    ('4. Early stopping (best)', mc4, mp4, '5000 clips, val-stopped at epoch 2.\nThe winner.', '#c0ff3e'),
]

for idx, (title, mc, mp, subtitle, color) in enumerate(models):
    ax = fig.add_subplot(2, 2, idx+1, facecolor='#181828')
    ax.plot(mc, color='#3eff9e', linewidth=2, marker='o', markersize=5, label='Clean')
    ax.plot(mp, color='#ff3e7e', linewidth=2, marker='o', markersize=5, label='Perturbed')
    ax.axvline(7, color='#ffb13e', linestyle='--', alpha=0.4)

    ratio = mp[7] / max(mc[7], 1e-9)
    ax.set_title(f'{title}    →    {ratio:.1f}× spike',
                 color=color, fontsize=14, fontweight='bold', pad=10)
    ax.text(0.02, 0.95, subtitle, transform=ax.transAxes,
            color='#aaa', fontsize=10, verticalalignment='top',
            family='monospace')

    ax.set_xlabel('Frame transition', color='#888')
    ax.set_ylabel('Mean surprise', color='#888')
    ax.tick_params(colors='#888')
    ax.legend(facecolor='#181828', edgecolor='#333', labelcolor='#ddd', loc='upper right')
    ax.grid(alpha=0.15)
    for spine in ax.spines.values():
        spine.set_color('#333')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Footer
fig.text(0.5, 0.01,
         'A deepfake detector built in one day on free Colab GPU. '
         'Same architecture in all four panels — only the training methodology changed.',
         ha='center', color='#888', fontsize=10, style='italic')

plt.savefig('day3_summary.png', dpi=150, facecolor='#0a0a0f', bbox_inches='tight')
plt.show()

print('\n✓ Saved to day3_summary.png — download via the Files panel')
print(f'\nSummary:')
print(f'  Synthetic only:        {mp1[7]/mc1[7]:.2f}× spike')
print(f'  Small real run:        {mp2[7]/mc2[7]:.2f}× spike')
print(f'  Big run (overtrained): {mp3[7]/mc3[7]:.2f}× spike')
print(f'  Early stopping (best): {mp4[7]/mc4[7]:.2f}× spike  ← the winner')

## 🎯 What to do next

Now that the entire pipeline works on synthetic data, here's the path to a real deepfake detector:

### Step 1: Replace the data
Swap `SyntheticVideoDataset` for one that loads real video clips. Options:
- **UCF-101** — 13k action videos, easy public download (~7 GB)
- **Kinetics-400 subset** — broader motion variety
- **FaceForensics++** — the deepfake gold standard (requires academic email request)

### Step 2: Scale up the architecture
Replace the tiny CNN encoder with a Vision Transformer Tiny (use `timm` library). Replace the MLP predictor with a small Transformer with causal masking. This brings you closer to the real LeWM at ~15M params.

### Step 3: Implement real SIGReg
The covariance regularizer here is a placeholder. The real SIGReg uses random projections and the Epps-Pulley statistical test for normality. Worth implementing once everything else works.

### Step 4: Move to Kaggle for serious training
Free Colab disconnects after ~12h. Kaggle gives you 30 hours of GPU per week with longer sessions. Same code, different runner.

### Step 5: Evaluate on real deepfakes
Once trained on real video, run surprise scoring on FaceForensics++ test set. Plot histograms of real vs fake surprise scores. Compute AUC. If AUC > 0.7 you have something interesting; > 0.85 you have a portfolio-grade result.

---

## 🔍 Reading list while you wait for things to train

- The original [LeWorldModel paper](https://arxiv.org/abs/2603.19312) (Maes et al., 2026)
- [I-JEPA paper](https://arxiv.org/abs/2301.08243) — the conceptual ancestor
- [DINO-WM paper](https://arxiv.org/abs/2411.04983) — the foundation-model competitor LeWM beats
- The [FaceForensics++ paper](https://arxiv.org/abs/1901.08971) — for understanding the deepfake benchmark

---

## 💡 If something didn't work

**No GPU detected:** Runtime menu → Change runtime type → T4 GPU.

**Out of memory:** Reduce `batch_size` in the DataLoader, or `n_clips` in the dataset.

**Surprise spike isn't visible:** Train for more epochs (try 30–50), or increase `embed_dim` to 128.

**Training collapses (pred goes to 0 immediately):** Increase `lam` to 5.0 or 10.0.

Good luck — and ping me when you have results to discuss!